## Load Silver Data

In [0]:
from pyspark.sql.functions import (
    col, lag, avg, stddev, when, lit, abs as spark_abs,
    min as spark_min, max as spark_max, sum as spark_sum,
    round as spark_round, count, datediff, expr
)
from pyspark.sql.window import Window

stock_prices = spark.table("raghavan_trading_signals.silver.stock_prices")
macro_wide = spark.table("raghavan_trading_signals.silver.macro_daily_wide")

print(f"Stock prices: {stock_prices.count():,} rows")
print(f"Macro data: {macro_wide.count():,} rows")

### Silver → Gold (Feature Engineering)
The Silver layer is clean but not production ready. They just have prices and macro values. \
The Gold layer **derives the features an ML model actually needs**.

| Feature Added | Explaination |
|---|---|
| **`daily_return`, `log_return`** | Percentage change in price from yesterday to today (e.g., +1.2% or −0.5%). | Models predict *change*, not absolute price. A stock at $200 going up 1% is the same "event" as a stock at $50 going up 1% — returns make different stocks directly comparable. |
| **`daily_range`** | How far apart the highest and lowest prices of the day were, as a % of the closing price. | A wide range means the day was choppy/uncertain. A narrow range means calm. Choppiness today often predicts choppiness tomorrow. |
| **`overnight_gap`** | How much the price "jumped" between yesterday's close and today's open (when the market was shut). | Big overnight jumps usually mean news broke after-hours (earnings, geopolitics). Models can learn that gappy openings behave differently from calm ones. |
| **`sma_7, 14, 21, 50, 100, 200`** | The average closing price over the last N days. Smooths out daily noise. | If today's price is *above* the average → the stock has been trending up. *Below* → trending down. Multiple time horizons (1 week vs 1 year) capture short-term vs long-term trends. |
| **`price_vs_sma_X`** | How far today's price is from the moving average, as a percentage. | "5% above its 50-day average" is a stronger signal than "above its 50-day average" — quantifying the gap helps the model weight it properly. |
| **`volatility_10d, 20d, 60d`** | How much the daily returns have been swinging around recently (the standard deviation). | High volatility = riskier, less predictable. The model uses this to know *how confident* it should be — a quiet stock's next move is easier to guess than a wild one's. |
| **`rsi_14`** (Relative Strength Index) | A 0–100 score of recent buying vs selling pressure over the last 14 days. Numbers near 0 → the stock has been heavily sold and might "bounce back." Numbers near 100 → it's been heavily bought and might "cool off." Helps the model spot extremes. | |
| **`macd_line`, `macd_signal`, `macd_histogram`** (Moving Average Convergence Divergence) | Compares a short-term average to a long-term average. The histogram = the gap between them. | When the short-term average crosses above the long-term one → momentum is building upward. The histogram tells the model *how strongly* and *when the trend is shifting*. |
| **`bb_upper`, `bb_lower`, `bb_middle`, `bb_width`, `bb_position`** (Bollinger Bands)| Draws a "normal range" channel around the price (middle = 20-day average, upper/lower = how wide a typical swing is). `bb_position` = where the price sits in that channel (0 = bottom, 1 = top). | Tells the model "is today's price unusually high or low relative to its own recent normal?" `bb_width` separately tells it whether the channel is widening (volatility rising) or tightening (calm). |
| **`volume_sma_10/20/50`, `volume_ratio`** | Average number of shares traded per day, and today's volume vs that average. | A price move on huge volume is "the market really means it." A move on tiny volume might be a fluke. The model learns to trust strong-volume signals more. |
| **All macro columns joined in** | VIX, interest rates, dollar strength, inflation, etc. Basically, the broader economy attached to every stock-day row. | A stock doesn't move in isolation. Apple on a day the Fed raises rates behaves differently than Apple on a calm day. Macro context gives the model the "weather conditions." |
| **`yield_curve_slope`** | The difference between long-term and short-term government bond rates. When it goes negative ("inverted"), recessions historically follow within a year. A single, well-known economic warning signal in one column. | |
| **`next_day_return`, `next_day_direction`** | Tomorrow's actual return / whether tomorrow was up (1) or down (0). This is the **answer key**. The model learns: "given today's features (left columns), here's what happened tomorrow." During training it sees the answer; in production it predicts it. | |
| **Drop rows where `sma_200` is null** | Removes the first ~200 days of each stock's history. Those early rows have incomplete moving averages (you can't compute a 200-day average from 50 days of data). Training on them would teach the model from garbage values. | |


## Daily Returns, Log Return, Daily Range and Overnight Gap

In [0]:
price_window = Window.partitionBy("symbol").orderBy("trade_date")

features = (
    stock_prices
    .withColumn("prev_close", lag("close", 1).over(price_window))
    .withColumn("daily_return",
        spark_round((col("close") - col("prev_close")) / col("prev_close"), 6)
    )
    .withColumn("daily_return_pct",
        spark_round(col("daily_return") * 100, 4)
    )
    .withColumn("log_return",
        spark_round(expr("ln(close / prev_close)"), 6)
    )
    # High-Low range as a volatility proxy
    .withColumn("daily_range",
        spark_round((col("high") - col("low")) / col("close"), 6)
    )
    # Gap: how much did the stock gap up/down from yesterday's close?
    .withColumn("overnight_gap",
        spark_round((col("open") - col("prev_close")) / col("prev_close"), 6)
    )
)

## Simple Moving Average (SMA) - sma_7, 14, 21, 50, 100, 200 and price_vs_sma_X

In [0]:
for period in [7, 14, 21, 50, 100, 200]:
    ma_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(
        -(period - 1), Window.currentRow
    )
    features = features.withColumn(
        f"sma_{period}",
        spark_round(avg("close").over(ma_window), 4)
    )
    features = features.withColumn(
        f"price_vs_sma_{period}",
        spark_round((col("close") - col(f"sma_{period}")) / col(f"sma_{period}"), 6)
    )

## Volatility (Rolling Standard Deviation) - volatility_10d, 20d, 60d

In [0]:
for period in [10, 20, 60]:
    vol_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(
        -(period - 1), Window.currentRow
    )
    features = features.withColumn(
        f"volatility_{period}d",
        spark_round(stddev("daily_return").over(vol_window), 6)
    )

## RSI (Relative Strength Index) - rsi_14

In [0]:
rsi_period = 14
rsi_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(
    -(rsi_period - 1), Window.currentRow
)

features = (
    features
    .withColumn("price_change", col("close") - col("prev_close"))
    .withColumn("gain", when(col("price_change") > 0, col("price_change")).otherwise(0))
    .withColumn("loss", when(col("price_change") < 0, spark_abs(col("price_change"))).otherwise(0))
    .withColumn("avg_gain", avg("gain").over(rsi_window))
    .withColumn("avg_loss", avg("loss").over(rsi_window))
    .withColumn("rs",
        when(col("avg_loss") == 0, lit(100))
        .otherwise(col("avg_gain") / col("avg_loss"))
    )
    .withColumn("rsi_14",
        spark_round(lit(100) - (lit(100) / (lit(1) + col("rs"))), 4)
    )
    .drop("price_change", "gain", "loss", "avg_gain", "avg_loss", "rs")
)

## MACD (Moving Average Convergence Divergence)
- MACD reveals changes in trend strength, direction, momentum, and duration.
- MACD Line = 12-day EMA - 26-day EMA
- Signal Line = 9-day EMA of MACD Line
- Histogram = MACD Line - Signal Line (positive = bullish momentum)

In [0]:
ema_12_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-11, Window.currentRow)
ema_26_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-25, Window.currentRow)

features = (
    features
    .withColumn("ema_12_approx", spark_round(avg("close").over(ema_12_window), 4))
    .withColumn("ema_26_approx", spark_round(avg("close").over(ema_26_window), 4))
    .withColumn("macd_line",
        spark_round(col("ema_12_approx") - col("ema_26_approx"), 4)
    )
)

macd_signal_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-8, Window.currentRow)
features = features.withColumn(
    "macd_signal",
    spark_round(avg("macd_line").over(macd_signal_window), 4)
)
features = features.withColumn(
    "macd_histogram",
    spark_round(col("macd_line") - col("macd_signal"), 4)
)

features = features.drop("ema_12_approx", "ema_26_approx")

## Bollinger Bands
- Bollinger Bands = SMA(20) ± 2 * StdDev(20).
- When price touches upper band = potentially overbought.
- When price touches lower band = potentially oversold.
- Band width indicates volatility - `bb_upper`, `bb_lower`, `bb_middle`, `bb_width`, `bb_position`

In [0]:
bb_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-19, Window.currentRow)

features = (
    features
    .withColumn("bb_middle", spark_round(avg("close").over(bb_window), 4))
    .withColumn("bb_std", stddev("close").over(bb_window))
    .withColumn("bb_upper", spark_round(col("bb_middle") + (lit(2) * col("bb_std")), 4))
    .withColumn("bb_lower", spark_round(col("bb_middle") - (lit(2) * col("bb_std")), 4))
    .withColumn("bb_width",
        spark_round((col("bb_upper") - col("bb_lower")) / col("bb_middle"), 6)
    )
    .withColumn("bb_position",
        spark_round(
            (col("close") - col("bb_lower")) / (col("bb_upper") - col("bb_lower")),
            4
        )
    )
    .drop("bb_std")
)

## Volume Features 
For example, `volume_sma_10/20/50`, `volume_ratio`

In [0]:
for period in [10, 20, 50]:
    vol_ma_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(
        -(period - 1), Window.currentRow
    )
    features = features.withColumn(
        f"volume_sma_{period}",
        spark_round(avg("volume").over(vol_ma_window), 0)
    )

features = features.withColumn(
    "volume_ratio",
    spark_round(col("volume") / col("volume_sma_20"), 4)
)

## Join Macro Data

In [0]:
features = features.join(
    macro_wide,
    features["trade_date"] == macro_wide["indicator_date"],
    "left"
).drop("indicator_date")

# Compute yield curve slope (10Y - 2Y) — a key recession predictor
features = features.withColumn(
    "yield_curve_slope",
    spark_round(col("treasury_10y") - col("treasury_2y"), 4)
)

## Create the Target Label
- This is what the ML model will predict:
- Did the stock go UP or DOWN the NEXT day?

In [0]:
features = features.withColumn(
    "next_day_return",
    spark_round(lag("daily_return", -1).over(price_window), 6)
)
features = features.withColumn(
    "next_day_direction",
    when(col("next_day_return") > 0, lit(1)).otherwise(lit(0))
)

## Drop Warmup Rows and Write to Gold
- The first ~200 rows per stock have incomplete moving averages (not enough history).
- Drop them so the ML model doesn't train on garbage.

In [0]:
features_clean = (
    features
    .filter(col("sma_200").isNotNull())
    .filter(col("next_day_direction").isNotNull())
    .orderBy("symbol", "trade_date")
)

print(f"Gold feature table: {features_clean.count():,} rows")
print(f"Columns: {len(features_clean.columns)}")

# Write to Gold as a managed Delta table
(
    features_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("raghavan_trading_signals.gold.daily_features")
)

print("Gold table written successfully!")

## Validation of the Gold table

In [0]:
gold_df = spark.table("raghavan_trading_signals.gold.daily_features")

print(f"Total rows: {gold_df.count():,}")
print(f"Total columns: {len(gold_df.columns)}")
print(f"Stocks: {gold_df.select('symbol').distinct().count()}")
print(f"Date range: {gold_df.agg({'trade_date': 'min'}).collect()[0][0]} to "
      f"{gold_df.agg({'trade_date': 'max'}).collect()[0][0]}")

# Show sample for one stock
display(
    gold_df
    .filter(col("symbol") == "AAPL")
    .orderBy(col("trade_date").desc())
    .select("symbol", "trade_date", "close", "rsi_14", "macd_line",
            "bb_position", "volatility_20d", "volume_ratio",
            "vix", "yield_curve_slope", "next_day_direction")
    .limit(10)
)

## Gold feature table — what we ended up with

- **66,550 rows** survived the warm-up filter: 50 stocks × ~1,330 trading days each, minus the first
  ~200 days per stock that have an incomplete `sma_200`, minus the last day per stock that has no
  `next_day_direction` label.
- **56 columns total = 41 model features + 14 metadata/price columns + 1 label** (`next_day_direction`).
- **50 stocks, 2020-01-02 → 2025-04-17.**
- Each row = one stock on one day with returns, volatility, momentum (RSI/MACD), Bollinger, volume,
  and macro context features, plus the answer key. This single table feeds both the AutoML and LSTM
  notebooks, and (later) batch scoring and the serving endpoint.